# HireMind AI - Resume Model Training Pipeline

This notebook trains a resume classification model using the cleaned dataset to predict candidate categories/roles.

In [ ]:
import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Paths
PROCESSED_DATA_PATH = os.path.join("..", "data", "processed", "cleaned_resumes.csv")
MODEL_SAVE_PATH = os.path.join("resume_classifier.pkl")

## 1. Load Processed Dataset

In [ ]:
df = pd.read_csv(PROCESSED_DATA_PATH)
df = df.dropna(subset=["Resume_str", "Category"])
print(f"Loaded {df.shape[0]} valid rows for model training.")
print("Categories distribution:")
print(df["Category"].value_counts().head(10))

## 2. Train-Test Split

In [ ]:
X = df["Resume_str"]
y = df["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

## 3. Train Pipeline (TF-IDF + Classifier)

We construct a machine learning pipeline that extracts text features using TF-IDF and trains a Naive Bayes model to predict categories.

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")),
    ("classifier", MultinomialNB(alpha=0.1))
])

print("Training the classifier pipeline...")
pipeline.fit(X_train, y_train)
print("Model training complete!")

## 4. Evaluate Model Performance

In [ ]:
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

## 5. Save Model Pipeline

Save the trained classification model pipeline to a serialized pickle file.

In [ ]:
with open(MODEL_SAVE_PATH, "wb") as f:
    pickle.dump(pipeline, f)
print(f"Successfully saved classifier model pipeline to {MODEL_SAVE_PATH}")